In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd



# Go up one directory to the project root and add it to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.preprocessing  import load_dataset, add_financial_features, add_additional_financial_features

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "application_train.csv"


df = load_dataset(DATA_PATH)

df_features = add_financial_features(df)

print("Original shape:", df.shape)
print("Feature-engineered shape:", df_features.shape)

df_features.head()

Original shape: (307511, 122)
Feature-engineered shape: (307511, 125)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,AGE_YEARS
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0.0,0.0,0.0,0.0,0.0,1.0,2.007889,0.121978,25.902806
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,4.790750,0.132217,45.900068
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,2.000000,0.100000,52.145106
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,NaN,NaN,NaN,NaN,NaN,NaN,2.316167,0.219900,52.032854
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,4.222222,0.179963,54.570842


In [3]:
df_features_v2 = add_additional_financial_features(df_features)

print("Before:", df_features.shape)
print("After:", df_features_v2.shape)

Before: (307511, 125)
After: (307511, 131)


In [4]:
X = df_features.drop(
    columns=["TARGET", "SK_ID_CURR"]
)

y = df_features["TARGET"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (307511, 123)
Target shape: (307511,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.20,
    random_state = 42,
    stratify = y,
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(4))

Training features: (246008, 123)
Testing features: (61503, 123)

Training target distribution:
TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64

Testing target distribution:
TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64


In [6]:
extra_features = [
    "CREDIT_INCOME_RATIO_V2",
    "GOODS_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO_V2",
    "EMPLOYED_YEARS",
    "AGE_YEARS_V2",
    "CREDIT_GOODS_RATIO",
]

# Rebuild the feature sets using the same original split indices.
X_train_v2 = X_train.copy()
X_test_v2 = X_test.copy()

# Add the engineered features using the matching row indices.
X_train_v2[extra_features] = (
    df_features_v2.loc[X_train.index, extra_features]
)

X_test_v2[extra_features] = (
    df_features_v2.loc[X_test.index, extra_features]
)

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


def build_logistic_model(X_train_data):
    numeric_features = X_train_data.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_features = X_train_data.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter=1000, random_state=42),
        ),
    ])

    return model

In [8]:
model_a = build_logistic_model(X_train)

model_a.fit(X_train, y_train)

pred_a = model_a.predict(X_test)
proba_a = model_a.predict_proba(X_test)[:, 1]

C:\Users\nandi\AppData\Local\Temp\ipykernel_10640\323386268.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train_data.select_dtypes(


In [9]:
model_b = build_logistic_model(X_train_v2)

model_b.fit(X_train_v2, y_train)

pred_b = model_b.predict(X_test_v2)
proba_b = model_b.predict_proba(X_test_v2)[:, 1]

C:\Users\nandi\AppData\Local\Temp\ipykernel_10640\323386268.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train_data.select_dtypes(


In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


def evaluate_model(name, y_true, y_pred, y_proba):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "ROC_AUC": roc_auc_score(y_true, y_proba),
        "Average_Precision": average_precision_score(
            y_true, y_proba
        ),
    }


comparison = pd.DataFrame([
    evaluate_model("Model A — Baseline", y_test, pred_a, proba_a),
    evaluate_model("Model B — Engineered", y_test, pred_b, proba_b),
])

comparison.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Average_Precision
0,Model A — Baseline,0.9195,0.5702,0.0131,0.0256,0.7483,0.2274
1,Model B — Engineered,0.9195,0.5625,0.0145,0.0283,0.7495,0.2317


In [11]:
feature_names = (
    model_b.named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    model_b.named_steps["classifier"]
    .coef_[0]
)

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})

feature_importance["absolute_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance.sort_values(
    "absolute_coefficient",
    ascending=False
).head(20)

,feature,coefficient,absolute_coefficient
233,cat__ORGANIZATION_TYPE_Transport: type 3,0.651840,0.651840
210,cat__ORGANIZATION_TYPE_Military,-0.577750,0.577750
4,num__AMT_GOODS_PRICE,-0.562740,0.562740
2,num__AMT_CREDIT,0.499321,0.499321
215,cat__ORGANIZATION_TYPE_Realtor,0.493489,0.493489
29,num__EXT_SOURCE_3,-0.468724,0.468724
115,cat__CODE_GENDER_F,-0.423999,0.423999
119,cat__FLAG_OWN_CAR_Y,-0.396106,0.396106
28,num__EXT_SOURCE_2,-0.383240,0.383240
220,cat__ORGANIZATION_TYPE_Security Ministries,-0.379671,0.379671
